In [3]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.master('local[*]').getOrCreate()

In [8]:
ordItems = spark.read.csv('/content/drive/MyDrive/Spark_Practice/RetailDB_SalesData/Order_items/part-00000', ('id int, order_id int, product_id int, quantity int, subtotal float, price float'))
ordItems.show(5)
ordItems.printSchema()

+---+--------+----------+--------+--------+------+
| id|order_id|product_id|quantity|subtotal| price|
+---+--------+----------+--------+--------+------+
|  1|       1|       957|       1|  299.98|299.98|
|  2|       2|      1073|       1|  199.99|199.99|
|  3|       2|       502|       5|   250.0|  50.0|
|  4|       2|       403|       1|  129.99|129.99|
|  5|       4|       897|       2|   49.98| 24.99|
+---+--------+----------+--------+--------+------+
only showing top 5 rows
root
 |-- id: integer (nullable = true)
 |-- order_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- subtotal: float (nullable = true)
 |-- price: float (nullable = true)



### Summary

In [17]:
# For entire DF
ordItems.summary().show()

+-------+-----------------+------------------+-----------------+------------------+------------------+------------------+
|summary|               id|          order_id|       product_id|          quantity|          subtotal|             price|
+-------+-----------------+------------------+-----------------+------------------+------------------+------------------+
|  count|           172198|            172198|           172198|            172198|            172198|            172198|
|   mean|          86099.5| 34442.56682423721|660.4877176273824|2.1821275508426345|199.32066922046081|133.75906959048717|
| stddev|49709.42516431533|19883.325171992223|310.5144727900077|1.4663523175387168|112.74303987146766| 118.5589363325847|
|    min|                1|                 1|               19|                 1|              9.99|              9.99|
|    25%|            43033|             17204|              403|                 1|            119.98|              50.0|
|    50%|            860

In [19]:
# For specific columns
ordItems.select(ordItems.id, ordItems.product_id).summary().show()

+-------+-----------------+-----------------+
|summary|               id|       product_id|
+-------+-----------------+-----------------+
|  count|           172198|           172198|
|   mean|          86099.5|660.4877176273824|
| stddev|49709.42516431533|310.5144727900077|
|    min|                1|               19|
|    25%|            43033|              403|
|    50%|            86085|              627|
|    75%|           129134|             1004|
|    max|           172198|             1073|
+-------+-----------------+-----------------+



### Sum

In [34]:
from pyspark.sql.functions import sum, sum_distinct
ordItems.select(sum(ordItems.price), sum_distinct(ordItems.price)).show()

+--------------------+-------------------+
|          sum(price)|sum(DISTINCT price)|
+--------------------+-------------------+
|2.3033044265342712E7|   9832.41999053955|
+--------------------+-------------------+



### Count

In [35]:
from pyspark.sql.functions import count, count_distinct
ordItems.select(count(ordItems.price), count_distinct(ordItems.price)).show()

+------------+---------------------+
|count(price)|count(DISTINCT price)|
+------------+---------------------+
|      172198|                   57|
+------------+---------------------+



### First & Last

In [36]:
from pyspark.sql.functions import first, last
ordItems.sort(ordItems.price.asc()).select(first(ordItems.price), last(ordItems.price)).show()

+------------+-----------+
|first(price)|last(price)|
+------------+-----------+
|        9.99|    1999.99|
+------------+-----------+



### Collect as list & set
- `collect_list`: Collects all the data including duplicates
- `collect_set`: Collects all the distinct values

In [38]:
# Sample data

df = spark.createDataFrame([(1, 100), (2, 150), (3, 200), (4, 50), (5, 50)], ('id int, salary int'))
df.show()

+---+------+
| id|salary|
+---+------+
|  1|   100|
|  2|   150|
|  3|   200|
|  4|    50|
|  5|    50|
+---+------+



In [41]:
from pyspark.sql.functions import collect_list, collect_set
df.select(collect_list(df.salary), collect_set(df.salary)).show(truncate=False)

+-----------------------+-------------------+
|collect_list(salary)   |collect_set(salary)|
+-----------------------+-------------------+
|[100, 150, 200, 50, 50]|[50, 100, 150, 200]|
+-----------------------+-------------------+



### Skewness, Variance & Standard Deviation

In [43]:
from pyspark.sql.functions import skewness, variance, stddev
ordItems.select(skewness(ordItems.price), variance(ordItems.price), stddev(ordItems.price)).show()

+------------------+------------------+-----------------+
|   skewness(price)|   variance(price)|    stddev(price)|
+------------------+------------------+-----------------+
|1.5541175048553708|14056.221384313872|118.5589363325847|
+------------------+------------------+-----------------+

